In [ ]:
import pandas as pd

df = pd.read_csv("/content/Student_Performance.csv")

# Xóa các dòng trùng lặp
df = df.drop_duplicates().reset_index(drop=True)

# Tách biến đầu vào và biến mục tiêu
X = df.drop(columns=["Performance Index"])
y = df["Performance Index"]

print("Kích thước dữ liệu:", df.shape)
print("X:", X.shape)
print("y:", y.shape)

Kích thước dữ liệu: (9873, 6)
X: (9873, 5)
y: (9873,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (7898, 5)
X_test: (1975, 5)
y_train: (7898,)
y_test: (1975,)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "Hours Studied",
    "Previous Scores",
    "Sleep Hours",
    "Sample Question Papers Practiced"
]

categorical_features = [
    "Extracurricular Activities"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessor đã được tạo.")

Preprocessor đã được tạo.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "KNN": KNeighborsRegressor(n_neighbors=5)
}

print("Số mô hình:", len(models))
for name in models:
    print("-", name)

Số mô hình: 4
- Linear Regression
- Decision Tree
- Random Forest
- KNN


In [ ]:
from sklearn.pipeline import Pipeline

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

print("Đã tạo pipeline cho", len(pipelines), "mô hình.")

for name in pipelines:
    print("-", name)

Đã tạo pipeline cho 4 mô hình.
- Linear Regression
- Decision Tree
- Random Forest
- KNN


In [ ]:
import time

trained_models = {}
training_times = {}

for name, pipeline in pipelines.items():
    start_time = time.time()

    pipeline.fit(X_train, y_train)

    end_time = time.time()

    trained_models[name] = pipeline
    training_times[name] = end_time - start_time

    print(f"{name}: đã train xong ({training_times[name]:.4f} giây)")

Linear Regression: đã train xong (0.0474 giây)
Decision Tree: đã train xong (0.0495 giây)
Random Forest: đã train xong (1.4771 giây)
KNN: đã train xong (0.0195 giây)


In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)

    print(
        f"{name}: "
        f"{len(predictions[name])} predictions"
    )

Linear Regression: 1975 predictions
Decision Tree: 1975 predictions
Random Forest: 1975 predictions
KNN: 1975 predictions


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for name, y_pred in predictions.items():
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

print(results_df)

               Model       MAE      RMSE        R2
0  Linear Regression  1.646970  2.075066  0.988430
1      Decision Tree  2.433249  3.050595  0.974995
2      Random Forest  1.895022  2.369629  0.984912
3                KNN  2.361418  2.959169  0.976471


In [ ]:
prediction_times = {}

for name, model in trained_models.items():
    start_time = time.time()

    model.predict(X_test)

    end_time = time.time()

    prediction_times[name] = end_time - start_time

    print(
        f"{name}: "
        f"{prediction_times[name]:.6f} giây"
    )

Linear Regression: 0.007714 giây
Decision Tree: 0.009848 giây
Random Forest: 0.082161 giây
KNN: 0.024820 giây


In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")

start_time = time.time()

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_time = time.time() - start_time

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline - Mean Prediction")
print(f"MAE: {baseline_mae:.6f}")
print(f"RMSE: {baseline_rmse:.6f}")
print(f"R2: {baseline_r2:.6f}")
print(f"Time: {baseline_time:.6f} giây")

Baseline - Mean Prediction
MAE: 16.252553
RMSE: 19.301002
R2: -0.000978
Time: 0.001560 giây


In [ ]:
import joblib
import os
import tempfile

model_sizes = {}

for name, model in trained_models.items():
    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        temp_path = f.name

    joblib.dump(model, temp_path)

    size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    model_sizes[name] = size_mb

    os.remove(temp_path)

    print(f"{name}: {size_mb:.4f} MB")

Linear Regression: 0.0032 MB
Decision Tree: 0.8947 MB
Random Forest: 56.9174 MB
KNN: 0.9099 MB


In [ ]:
evaluation_results = results_df.copy()

evaluation_results["Training Time (s)"] = evaluation_results["Model"].map(training_times)
evaluation_results["Prediction Time (s)"] = evaluation_results["Model"].map(prediction_times)
evaluation_results["Model Size (MB)"] = evaluation_results["Model"].map(model_sizes)

evaluation_results = evaluation_results[
    [
        "Model",
        "MAE",
        "RMSE",
        "R2",
        "Training Time (s)",
        "Prediction Time (s)",
        "Model Size (MB)"
    ]
]

print(evaluation_results.to_string(index=False))

            Model      MAE     RMSE       R2  Training Time (s)  Prediction Time (s)  Model Size (MB)
Linear Regression 1.646970 2.075066 0.988430           0.047444             0.007714         0.003213
    Decision Tree 2.433249 3.050595 0.974995           0.049520             0.009848         0.894656
    Random Forest 1.895022 2.369629 0.984912           1.477097             0.082161        56.917398
              KNN 2.361418 2.959169 0.976471           0.019472             0.024820         0.909859


In [ ]:
# Lưu bảng kết quả đánh giá sơ bộ
evaluation_results.to_csv(
    "/content/training_results.csv",
    index=False
)

print("Đã lưu training_results.csv")

Đã lưu training_results.csv


## Kết luận huấn luyện mô hình

Trong bước huấn luyện, 4 mô hình hồi quy được xây dựng và đánh giá trên cùng một tập dữ liệu:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- KNN Regressor

Tất cả các mô hình đều được xây dựng bằng Pipeline, trong đó preprocessing được thực hiện trước khi huấn luyện mô hình. Cách làm này giúp đảm bảo các bước xử lý dữ liệu được thực hiện nhất quán và hạn chế data leakage.

Dữ liệu được chia thành:
- Tập train: 7.898 mẫu
- Tập test: 1.975 mẫu

Các chỉ số được ghi nhận gồm:
- MAE
- RMSE
- R²
- Training Time
- Prediction Time
- Model Size

Ngoài 4 mô hình trên, một Baseline sử dụng giá trị trung bình của biến mục tiêu cũng được xây dựng để làm mốc so sánh.

Kết quả chi tiết được lưu trong file `training_results.csv`.

Việc lựa chọn mô hình cuối cùng sẽ được thực hiện trong bước đánh giá (`04_evaluate`) dựa trên nhiều tiêu chí thay vì chỉ dựa vào một metric duy nhất.